In [1]:
from google.colab import drive
drive.mount("/content/drive")

KeyboardInterrupt: 

In [2]:
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124 --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 236.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 411.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 253.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 419.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 274.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 360.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 247.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 337.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 150.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 383.8 MB/s eta 0:00:00


# TRAINING

In [3]:
############################################################################################
#                                  TRAINING CONFIGURATION                                  #
############################################################################################
"""
List of models available for training:
https://DOCS.unsloth.ai/get-started/all-our-models
"""
import os
from typing import List, Optional, Literal
from pydantic_settings import BaseSettings


class TrainingSettings(BaseSettings):

    # MODEL PARAMETERS
    MODEL_NAME: str = "unsloth/Qwen2.5-0.5B-Instruct"                    # Must be available on unsloth
    MAX_SEQ_LENGTH: int = 8000                                           # Choose any! We auto support RoPE Scaling internally!
    DTYPE: Optional[Literal["float16", "bfloat16"]] = None               # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

    # PEFT MODEL PARAMETERS
    R: int = 32                                                          # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    TARGET_MODULES: List[str] = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",]
    LORA_ALPHA: int = 16                                                 # Scaling factor for LoRA updates. Higher values increase LoRA influence on the base model
    LORA_DROPOUT: int = 0                                                # Supports any, but = 0 is optimized
    BIAS: str = "none"                                                   # Supports any, but = "none" is optimized
    USE_GRADIENT_CHECKPOINTING: bool | Literal["unsloth"] = "unsloth"    # True or "unsloth" for very long context
    RANDOM_STATE: int = 3407                                             # Random seed for LoRA initialization and reproducibility
    USE_RSLORA: bool = False                                             # We support rank stabilized LoRA
    LOFTQ_CONFIG: Optional[dict] = None                                                  # And LoftQ

    # DATASET
    CHAT_TEMPLATE: str = "qwen-2.5"                                      # Chat formatting template used to structure conversations for training.
    DATASET_PATH: str = "/content/drive/MyDrive/Procern-Training/dataset_train_category_prediction_v2.json"                          # Path to the training dataset file.
    DATASET_SHUFFLE_SEED: int = 65                                       # Random seed used when shuffling the dataset before training.
    SPLIT_SHUFFLE_SEED: int = 42                                         # Random seed used when splitting dataset into train/validation sets.
    VALIDATION_SPLIT_SIZE: float = 0.0362647325475975                                   # Fraction of dataset reserved for validation.

    # TRAINING PARAMETERS
    PER_DEVICE_TRAIN_BATCH_SIZE: int = 4                                 # Number of training examples processed by each device (GPU/CPU) in one forward/backward pass.
    PER_DEVICE_EVAL_BATCH_SIZE: int = 4                                  # Number of evaluation examples processed per device in one forward pass. Lower values reduce memory usage during evaluation.
    GRADIENT_ACCUMULATION_STEPS: int = 8                                 # Effective batch size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    EVAL_STRATEGY: str = "steps"                                         # Evaluation frequency mode: either after fixed steps or every epoch.
    EVAL_STEPS: int = 300                                                 # Run evaluation every N training steps when using "steps" strategy.
    EVAL_ACCUMULATION_STEPS: int = 2                                     # Number of steps to accumulate evaluation data before processing. Helps manage memory during evaluation without affecting training.
    WARMUP_STEPS: int = 25                                               # Number of steps to gradually increase the learning rate from 0 to the initial value (LEARNING_RATE).
    NUM_TRAIN_EPOCHS: int = 3                                            # Total number of passes over the entire dataset.
    # MAX_STEPS: int = 3                                                 # Disable if using 'num_train_epochs' otherwise this will overwrite it.
    LEARNING_RATE: float = 2e-4                                          # Initial step size for updating model parameters. Smaller values ensure slower but stable convergence.
    OPTIM: str = "adamw_8bit"                                            # Optimizer type for updating model weights. "adamw_8bit" usually works for most cases.
    WEIGHT_DECAY: float = 0.01                                           # Regularization parameter to prevent overfitting by penalizing large weights.
    LR_SCHEDULER_TYPE: str = "linear"                                    # Type of learning rate schedule. "linear" means the learning rate decreases linearly from its initial value to 0 over the course of training.
    TRAINING_SEED: int = 3407                                            # Random seed for reproducibility. Ensures consistent results when re-running the training script.
    REPORT_TO: str = "none"                                              # Reporting framework for logging metrics (e.g., "wandb", "tensorboard"). "none" disables reporting.
    LOGGING_STRATEGY: Literal["epoch", "steps"] = "steps"                # 'epoch' or 'steps'.
    LOGGING_STEPS: int = 300                                              # Frequency (in steps/epoch) of logging training metrics like loss.
    SAVE_STRATEGY: Literal["epoch", "steps", "no"] = "no"                   # 'epoch' or 'steps' or 'no'.
    SAVE_STEPS: int = 1                                                  # Frequency (in steps/epoch) of saving the merged model.
    TRAIN_ON_RESPONSES_ONLY: bool = True                                 # Only train on the assistants output and ignore the loss on user's input.
    LOAD_IN_4BIT: bool = False                                           # Use 4bit quantization to reduce memory usage. Can be False.
    INSTRUCTION_PART: str = "<|im_start|>user\n"                         # The part of the chat template that corresponds to the user's input.
    RESPONSE_PART: str = "<|im_start|>assistant\n"                       # The part of the chat template that corresponds to the assistant's output.


    # OUTPUT
    MODEL_STORE_DIR: str = "/content/drive/MyDrive/Procern-Training/models/Category-Prediction-V2"                       # Store all model versions here.
    SAVE_16BITS: bool = True                                             # Merged model saved in float16.
    SAVE_Q6_K: bool = False                                               # First quantised version. Uses Q8_K for all tensors".
    SAVE_QUANTIZED: bool = False                                         # Second quantised version. Recommended. Slow conversion. Fast inference, small files.


print("Loading settings ...")
settings = TrainingSettings()
print(f"Training Settings loaded: {settings}")

Loading settings ...
Training Settings loaded: MODEL_NAME='unsloth/Qwen2.5-0.5B-Instruct' MAX_SEQ_LENGTH=8000 DTYPE=None R=32 TARGET_MODULES=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] LORA_ALPHA=16 LORA_DROPOUT=0 BIAS='none' USE_GRADIENT_CHECKPOINTING='unsloth' RANDOM_STATE=3407 USE_RSLORA=False LOFTQ_CONFIG=None CHAT_TEMPLATE='qwen-2.5' DATASET_PATH='/content/drive/MyDrive/Procern-Training/dataset_train_category_prediction_v2.json' DATASET_SHUFFLE_SEED=65 SPLIT_SHUFFLE_SEED=42 VALIDATION_SPLIT_SIZE=0.0362647325475975 PER_DEVICE_TRAIN_BATCH_SIZE=4 PER_DEVICE_EVAL_BATCH_SIZE=4 GRADIENT_ACCUMULATION_STEPS=2 EVAL_STRATEGY='steps' EVAL_STEPS=300 EVAL_ACCUMULATION_STEPS=2 WARMUP_STEPS=25 NUM_TRAIN_EPOCHS=3 LEARNING_RATE=0.0002 OPTIM='adamw_8bit' WEIGHT_DECAY=0.01 LR_SCHEDULER_TYPE='linear' TRAINING_SEED=3407 REPORT_TO='none' LOGGING_STRATEGY='steps' LOGGING_STEPS=300 SAVE_STRATEGY='no' SAVE_STEPS=1 TRAIN_ON_RESPONSES_ONLY=True LOAD_IN_4BIT=False INSTRUCTIO

In [ ]:
# TRAINING.PY
"""
COMMANDS TO RUN BEFORE TRAINING:

pip install unsloth "xformers==0.0.28.post2"
pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


NOTES:

1. On T4 GPU, if we dont manually reduce 'per_device_train_batch_size'
from 2 to 1, we run a risk of the notebook crashing while training is
going on.

2. Once the training is over, the script automatically stores both the
adapters as well as gguf_version of the finetuned model. So make sure
sufficient space is available.
"""
import os
import time
from datetime import datetime
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
# from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

print("****************************************************************************************")
print("*                                   Unsloth Training                                   *")
print("****************************************************************************************")

# MODEL LOADING
model_loading_start_time = time.time()
print(f"Loading model {settings.MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = settings.MODEL_NAME,
    max_seq_length = settings.MAX_SEQ_LENGTH,
    dtype = settings.DTYPE,
    load_in_4bit = settings.LOAD_IN_4BIT
)
print(f"Model loaded in {time.time() - model_loading_start_time:.2f} seconds.")

# PEFT MODEL LOADING FOR TRAINING
peft_model_loading_start_time = time.time()
print("Loading PEFT Model ...")
model = FastLanguageModel.get_peft_model(
    model,
    r=settings.R,
    target_modules=settings.TARGET_MODULES,
    lora_alpha=settings.LORA_ALPHA,
    lora_dropout=settings.LORA_DROPOUT,
    bias=settings.BIAS,
    use_gradient_checkpointing=settings.USE_GRADIENT_CHECKPOINTING,
    random_state=settings.RANDOM_STATE,
    max_seq_length=settings.MAX_SEQ_LENGTH,
    use_rslora=settings.USE_RSLORA,
    loftq_config=settings.LOFTQ_CONFIG,
)
print(f"PEFT Model loaded in {time.time() - peft_model_loading_start_time:.2f} seconds.")

# DATASET LOADING AND PREPROCESSING
tokenizer = get_chat_template(
    tokenizer,
    chat_template = settings.CHAT_TEMPLATE,
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        ) for convo in convos
    ]

    # 🔧 Explicit tokenization — full control here!
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=settings.MAX_SEQ_LENGTH,
        padding=False,         # SFTTrainer handles padding via data collator
        return_tensors=None,   # Return plain Python lists, not tensors
    )

    return {
        "text": texts,
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
    }

dataset_loading_start_time = time.time()
print(f"Loading and preprocessing dataset from '{settings.DATASET_PATH}' ...")
dataset = load_dataset("json", data_files=settings.DATASET_PATH, split='train')
dataset = dataset.shuffle(seed=settings.DATASET_SHUFFLE_SEED)
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset_before_cols_removal = dataset    # TO BE USED FOR DEBUGGING LATER IF NEEDED
columns_to_remove = [col for col in dataset.column_names if col not in ['input_ids', 'attention_mask']]
dataset = dataset.remove_columns(columns_to_remove)
print(f"Removed the following columns from dataset: {columns_to_remove}")
print(f"Final dataset columns before splitting: {dataset.column_names}")
dataset = dataset.train_test_split(
    test_size=settings.VALIDATION_SPLIT_SIZE,
    shuffle=True,
    seed=settings.SPLIT_SHUFFLE_SEED
)
train_dataset = dataset['train']
eval_dataset = dataset['test']
print(f"Dataset loaded and preprocessed in "
            f"{time.time() - dataset_loading_start_time:.2f} seconds.")

# CREATING DIRECTORY STRUCTURE TO STORE ALL MODEL VERSIONS
model_store_dir = os.path.join(settings.MODEL_STORE_DIR, str(datetime.now()).replace(' ', '-'))
merged_16bit_dir = os.path.join(model_store_dir, "merged_16bit")
q6_k_dir = os.path.join(model_store_dir, "q6_k")
quantized_dir = os.path.join(model_store_dir, "quantized")
os.makedirs(merged_16bit_dir)
print(f"Directory created to store merged 16-bit model: '{merged_16bit_dir}'")
os.makedirs(q6_k_dir)
print(f"Directory created to store q6_k quantized model: '{q6_k_dir}'")
os.makedirs(quantized_dir)
print(f"Directory created to store quantized model: '{quantized_dir}'")

# INITIALISING TRAINING PARAMETERS
trainer_init_start_time = time.time()
print("Initialising Trainer ...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        per_device_train_batch_size=settings.PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=settings.PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=settings.GRADIENT_ACCUMULATION_STEPS,
        eval_strategy=settings.EVAL_STRATEGY,
        eval_steps=settings.EVAL_STEPS,
        eval_accumulation_steps=settings.EVAL_ACCUMULATION_STEPS,
        max_seq_length=settings.MAX_SEQ_LENGTH,
        warmup_steps=settings.WARMUP_STEPS,
        num_train_epochs=settings.NUM_TRAIN_EPOCHS,
        # max_steps=settings.MAX_STEPS,    # DISABLE IF USING 'num_train_epochs' OTHERWISE THIS WILL OVERWRITE IT.
        learning_rate=settings.LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim=settings.OPTIM,
        weight_decay=settings.WEIGHT_DECAY,
        lr_scheduler_type=settings.LR_SCHEDULER_TYPE,
        seed=settings.TRAINING_SEED,
        output_dir=model_store_dir,
        report_to=settings.REPORT_TO,
        logging_strategy=settings.LOGGING_STRATEGY,
        logging_steps=settings.LOGGING_STEPS,
        save_strategy=settings.SAVE_STRATEGY,
        save_steps=settings.SAVE_STEPS
    ),
)
print(f"Trainer initialised in {time.time() - trainer_init_start_time:.2f} seconds.")

# TRAINING
if settings.TRAIN_ON_RESPONSES_ONLY:
    trainer = train_on_responses_only(
        trainer,
        instruction_part=settings.INSTRUCTION_PART,
        response_part=settings.RESPONSE_PART,
    )
training_start_time = time.time()
print("Starting training ...")
trainer_stats = trainer.train()    # ADAPTERS ARE SAVED DURING THIS STEP
print(f"Training completed in {time.time() - training_start_time:.2f} seconds.")

# SAVING FULL MODEL (16_bits)
if settings.SAVE_16BITS:
    model_saving_start_time = time.time()
    print("Saving merged model in 16-bit format ...")
    model.save_pretrained_merged(merged_16bit_dir, tokenizer, save_method="merged_16bit",)
    print(f"Merged model saved in 16-bit format in "
                f"{time.time() - model_saving_start_time:.2f} seconds.")

# SAVING 'Q6_K' VERSION
if settings.SAVE_Q6_K:
    model_saving_start_time = time.time()
    print("Saving merged model in q6_k quantized format ...")
    model.save_pretrained_gguf(q6_k_dir, tokenizer, quantization_method="q6_k")
    print(f"Merged model saved in q6_k quantized format in "
                f"{time.time() - model_saving_start_time:.2f} seconds.")

# SAVING 'QUANTIZED' VERSION
if settings.SAVE_QUANTIZED:
    model_saving_start_time = time.time()
    print("Saving merged model in quantized format ...")
    model.save_pretrained_gguf(quantized_dir, tokenizer, quantization_method="quantized")
    print(f"Merged model saved in quantized format in "
                f"{time.time() - model_saving_start_time:.2f} seconds.")

print("*******************************************************************************")
print("*                                   The End                                   *")
print("*******************************************************************************")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
****************************************************************************************
*                                   Unsloth Training                                   *
****************************************************************************************
Loading model unsloth/Qwen2.5-0.5B-Instruct ...
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading b

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,574 | Num Epochs = 3 | Total steps = 9,966
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 17,596,416 of 511,629,184 (3.44% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
